# PhysNet training on MD17 / rMD17 data

Train PhysNetJax on a **bring-your-own** MD17 or revised MD17 (rMD17) NPZ file.

**Input keys (rMD17):** `nuclear_charges`, `coords`, `energies`, `forces`  
**Units:** Å, kcal/mol, kcal/mol/Å

**Splits:** official test indices are held out first (`rmd17/splits/index_test_*.csv`); train/valid are sampled only from `index_train_*.csv` (≤ 1000 structures recommended).

## 1. Configuration

In [ ]:
import os
from pathlib import Path

# --- point this at your NPZ file ---
DATA_PATH = Path("/scicore/home/meuwly/boitti0000/data/rmd17/npz_data/rmd17_aspirin.npz")

# Official rMD17 splits (Materials Cloud: rmd17/splits/index_{train,test}_0X.csv)
# None = auto-detect as <rmd17>/splits next to npz_data/
RMD17_SPLITS_DIR = None
SPLIT_ID = 1  # fold 01 .. 05

# Train/valid drawn only from official train pool (after test is held out)
N_TRAIN = 950
N_VALID = 50
SEED = 42

NUM_EPOCHS = 50
BATCH_SIZE = 16
LEARNING_RATE = 1e-3

MAX_STRUCTURES = None  # set e.g. 200 for a quick smoke test
CONVERT_TO_EV = True   # kcal/mol -> eV (matches PhysNetJax defaults)

CKPT_DIR = Path("checkpoints/rmd17_aspirin")
RUN_NAME = f"rmd17_aspirin_split{SPLIT_ID:02d}"
SAVE_EVERY_EPOCH = True

EVAL_DIR = CKPT_DIR / f"test_eval_split{SPLIT_ID:02d}"

os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.9")

## 2. Load rMD17 data

In [ ]:
import numpy as np
import jax

from mmml.data.rmd17 import load_rmd17_npz
from mmml.data.loaders import get_data_statistics

raw = np.load(DATA_PATH, allow_pickle=True)
print("NPZ keys:", list(raw.keys()))
for k in raw.keys():
    v = raw[k]
    print(f"  {k:20s} shape={getattr(v, 'shape', '?')}")

data = load_rmd17_npz(
    DATA_PATH,
    convert_to_ev=CONVERT_TO_EV,
    max_structures=MAX_STRUCTURES,
)

num_atoms = int(data["N"].max())
print(f"\nMolecule size: {num_atoms} atoms")
print(f"Loaded {len(data['E'])} structures")
print(get_data_statistics(data))

## 3. Official test holdout, then train / validation

1. **Test** — indices from `index_test_{SPLIT_ID:02d}.csv` (never used for training).
2. **Train / valid** — random subset of `index_train_{SPLIT_ID:02d}.csv` only.

In [ ]:
def subset_arrays(d, idx):
    out = {}
    for key, value in d.items():
        if isinstance(value, np.ndarray) and value.shape[0] == n_total:
            out[key] = value[idx]
        else:
            out[key] = value
    return out


from mmml.data.rmd17 import load_rmd17_official_splits, resolve_rmd17_splits_dir

splits_dir = resolve_rmd17_splits_dir(DATA_PATH, RMD17_SPLITS_DIR)
official_train_pool, test_idx = load_rmd17_official_splits(splits_dir, SPLIT_ID)

n_total = len(data["E"])
assert official_train_pool.max() < n_total and test_idx.max() < n_total, (
    "Split indices out of range for loaded NPZ"
)
assert N_TRAIN + N_VALID <= len(official_train_pool), (
    f"Need {N_TRAIN + N_VALID} train-pool samples, "
    f"official fold has {len(official_train_pool)}"
)

# 1) Hold out official test set first
test_data = subset_arrays(data, test_idx)

# 2) Train / valid from official train pool only
rng = np.random.RandomState(SEED)
perm = rng.permutation(len(official_train_pool))
train_idx = official_train_pool[perm[:N_TRAIN]]
valid_idx = official_train_pool[perm[N_TRAIN : N_TRAIN + N_VALID]]

train_data = subset_arrays(data, train_idx)
valid_data = subset_arrays(data, valid_idx)

assert not (set(train_idx) & set(test_idx))
assert not (set(valid_idx) & set(test_idx))

print(f"Splits dir: {splits_dir}")
print(f"Official fold {SPLIT_ID:02d}: train pool={len(official_train_pool)}, test={len(test_idx)}")
print(f"Train: {len(train_data['E'])}  Valid: {len(valid_data['E'])}  Test: {len(test_data['E'])}")

## 4. Model & training

In [ ]:
from mmml.physnetjax.physnetjax.models.model import EF
from mmml.physnetjax.physnetjax.training.training import train_model

print("JAX devices:", jax.devices())

key = jax.random.PRNGKey(SEED)

model = EF(
    features=64,
    max_degree=0,
    num_iterations=3,
    num_basis_functions=32,
    cutoff=5.0,
    natoms=num_atoms,
    max_atomic_number=int(data["Z"].max()),
    charges=False,
    zbl=False,
)

ema_params, best_loss = train_model(
    key=key,
    model=model,
    train_data=train_data,
    valid_data=valid_data,
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    num_atoms=num_atoms,
    energy_weight=1.0,
    forces_weight=52.91,
    data_keys=("R", "Z", "F", "E", "N"),
    name=RUN_NAME,
    ckpt_dir=CKPT_DIR,
    best=True,
    save_every_epoch=SAVE_EVERY_EPOCH,
    batch_method="default",
    log_tb=False,
    print_freq=1,
)

run_dirs = sorted(CKPT_DIR.glob(f"{RUN_NAME}-*"))
RUN_CKPT_DIR = run_dirs[-1] if run_dirs else None

print(f"\nDone. Best validation objective: {best_loss:.6f}")
print(f"Checkpoints ({NUM_EPOCHS} epochs): {RUN_CKPT_DIR}")

## 5. Official test-set evaluation

Evaluate on the held-out **official** test indices (`index_test_{SPLIT_ID:02d}.csv`). Metrics are in kcal/mol (same convention as `mmml physnet-evaluate`).

In [ ]:
import json
import matplotlib.pyplot as plt

from mmml.physnetjax.physnetjax.data.batches import prepare_batches_jit
from mmml.physnetjax.physnetjax.analysis.analysis import plot_stats

key_eval, _ = jax.random.split(jax.random.PRNGKey(SEED + 1))
test_batches = prepare_batches_jit(
    key_eval,
    test_data,
    BATCH_SIZE,
    data_keys=("R", "Z", "F", "E", "N"),
    num_atoms=num_atoms,
)

print(f"Test batches: {len(test_batches)} × batch_size={BATCH_SIZE}")

stats = plot_stats(
    test_batches,
    model,
    ema_params,
    _set=f"Test | {RUN_NAME}",
    do_kde=False,
    batch_size=BATCH_SIZE,
    do_plot=True,
)

EV_PER_KCAL = 0.043364106370
metrics = {
    "n_test": int(len(test_data["E"])),
    "split_id": SPLIT_ID,
    "splits_dir": str(splits_dir),
    "checkpoint_run": str(RUN_CKPT_DIR) if RUN_CKPT_DIR else None,
    "energy": {
        "mae_kcal_mol": float(stats["E_mae"]),
        "rmse_kcal_mol": float(stats["E_rmse"]),
        "mae_ev": float(stats["E_mae"]) * EV_PER_KCAL,
        "rmse_ev": float(stats["E_rmse"]) * EV_PER_KCAL,
    },
    "forces": {
        "mae_kcal_mol": float(stats["F_mae"]),
        "rmse_kcal_mol": float(stats["F_rmse"]),
        "mae_ev": float(stats["F_mae"]) * EV_PER_KCAL,
        "rmse_ev": float(stats["F_rmse"]) * EV_PER_KCAL,
    },
}

EVAL_DIR.mkdir(parents=True, exist_ok=True)
with open(EVAL_DIR / "test_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

fig_path = EVAL_DIR / "parity_test.png"
plt.savefig(fig_path, dpi=200, bbox_inches="tight")
plt.show()

print(
    f"Energy  MAE={metrics['energy']['mae_kcal_mol']:.4f} kcal/mol  "
    f"({metrics['energy']['mae_ev']:.4f} eV)"
)
print(
    f"Forces  MAE={metrics['forces']['mae_kcal_mol']:.4f} kcal/mol/Å  "
    f"({metrics['forces']['mae_ev']:.4f} eV/Å)"
)
print(f"Wrote {EVAL_DIR / 'test_metrics.json'}")
print(f"Wrote {fig_path}")

## 6. (Optional) Save converted MMML NPZ

In [ ]:
SAVE_CONVERTED = False

if SAVE_CONVERTED:
    out_dir = Path("rmd17_converted")
    out_dir.mkdir(exist_ok=True)
    np.savez(out_dir / "train.npz", **train_data)
    np.savez(out_dir / "valid.npz", **valid_data)
    np.savez(out_dir / "test.npz", **test_data)
    print("Saved:", out_dir / "train.npz", out_dir / "valid.npz", out_dir / "test.npz")